# Chuỗi thời gian và text: hai quy tắc quan trọng

## Mục tiêu
- Split chuỗi thời gian theo thời gian, không shuffle.
- So sánh forecast với baseline naive.
- Biến text thành feature bằng TF-IDF trong pipeline.

In [ ]:
import numpy as np
import pandas as pd

dates = pd.date_range('2025-01-01', periods=90, freq='D')
rng = np.random.default_rng(42)
series = pd.Series(100 + np.arange(90) * .4 + rng.normal(0, 4, 90), index=dates)
train, test = series.iloc[:-14], series.iloc[-14:]
naive_forecast = pd.Series(train.iloc[-1], index=test.index)
mae = (test - naive_forecast).abs().mean()
mae

## Time series hygiene
Feature tại ngày *t* chỉ dùng thông tin biết được tại hoặc trước *t*. Random split tạo leak tương lai về quá khứ. Baseline seasonal naive thường mạnh hơn dự đoán của người mới bắt đầu.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

texts = ['sản phẩm rất tốt', 'giao hàng quá chậm', 'tôi hài lòng', 'dịch vụ tệ']
labels = [1, 0, 1, 0]
text_model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression())
text_model.fit(texts, labels)
text_model.predict(['giao hàng tốt'])

## Text model limitations
TF-IDF không hiểu ngữ cảnh, phủ định hay sarcasm tốt; example nhỏ ở đây chỉ minh hoạ API, không phải model production. Với tiếng Việt, quyết định tokenizer, normalization, privacy và nhãn phải được kiểm chứng trên data thật.

## Tự luyện
1. So sánh naive forecast với moving average 7 ngày.
2. Liệt kê ba feature time series có nguy cơ leakage.
3. Thu thập 20 câu text có quyền sử dụng, label thủ công và đánh giá hold-out.

**Checkpoint:** Bạn biết vì sao không dùng shuffle split cho forecast.